# Legal Contract RAG - Reference Solution

TF-IDF chunk retrieval + extractive span selection for legal contract question answering.

In [ ]:
import csv, re, string
from collections import Counter

def load_csv(path):
    with open(path,'r',encoding='utf-8') as f: return list(csv.DictReader(f))

def tokenize(text):
    return text.lower().translate(str.maketrans('','',string.punctuation)).split()

def chunk_text(text, size=100, overlap=20):
    words = text.split()
    return [' '.join(words[i:i+size]) for i in range(0, len(words), size-overlap)]

def best_chunk(question, contract_text):
    q_toks = set(tokenize(question))
    chunks = chunk_text(contract_text)
    return max(chunks, key=lambda c: sum(Counter(tokenize(c))[t] for t in q_toks))

contracts = {c['contract_id']: c['text'] for c in load_csv('dataset/public/contracts.csv')}
questions = load_csv('dataset/public/test_questions.csv')
print(f'Contracts: {len(contracts)} | Questions: {len(questions)}')

In [ ]:
results = []
for q in questions:
    chunk = best_chunk(q['question'], contracts.get(q['contract_id'],''))
    sentences = [s.strip() for s in re.split(r'[.!?\n]', chunk) if len(s.strip()) > 10]
    q_toks = set(tokenize(q['question']))
    answer = max(sentences, key=lambda s: sum(Counter(tokenize(s))[t] for t in q_toks), default=chunk[:200])
    results.append({'question_id': q['question_id'], 'predicted_answer': answer})

with open('predictions.csv','w',newline='') as f:
    w = csv.DictWriter(f,fieldnames=['question_id','predicted_answer'])
    w.writeheader(); w.writerows(results)
print(f'Saved {len(results)} predictions')